# Lab 2: Local Search

In this lab you will:

1. Formulate n-queens for local search: a state, a cost, and a set of neighbors.
2. Implement hill climbing, hill climbing with sideways moves, and random restarts.
3. Implement simulated annealing.
4. Use code you've developed on a very different problem: the travelling salesperson problem.
5. Compare the algorithms fairly at an equal budget, and see how that depends on how we count the budget.


**Reading:** Make sure to read sections 4.1-4.2 of the AIMA reading (file on canvas). There will be no journal entry required this week.

**Time:** The estimated time spent is about 4.5 hours. If you are spending significantly more than this, have a chat with a classmate or with the instructor to help get unstuck.
If you finish the lab within 4 hours, try the optional `lab2_extensions.ipynb` (will be released on Sep. 18).

**Setup:** you should have pandas installed, but see `README.md` for installation instructions if you haven't gotten it installed.

**What to submit:** commit this notebook with its saved outputs, plus the `results.csv` and `predictions.json` files that the notebook writes. Due Tuesday at 8:00 p.m, but since this notebook was released late I will offer free 1-day extensions if asked.

<div style="border-left: 4px solid #0072B2; padding: 0.4em 1em; background: #f4f8fb;">

### How to use this notebook

* **Stub cells** contain a function signature, a docstring, and `raise NotImplementedError`. Delete that line and write your code.
* **Check cells** are the cells after the stub cells that will run your code and check them. Run the checks and if there are errors go back and fix your code. Add `verbose=True` to any check to see a full traceback.
* **Hints** are in collapsible blocks. Try working first without using the hints. If you feel stuck, try clicking the triangle which will expand hints (roughly in order or increasing helpfulness).
* **Provided cells** are marked with a comment. You don't need to edit these, just run them. 

</div>

## Part 1: Setup and warm-up

*About 10 minutes. No coding.*

Run the next cell first. It checks that this notebook is running in the course environment with everything the lab needs.

In [ ]:
%run check_environment.py

Now the imports. Everything the lab provides lives in the `lab2kit` folder next to this notebook. You are welcome to open and read any of it.

In [ ]:
# PROVIDED
import math

import numpy as np
import pandas as pd

from lab2kit.problems import NQueens, TSP
from lab2kit.experiments import run_one, run_all
from lab2kit.predictions import record_predictions, reveal
from lab2kit.results import save_results
from lab2kit import viz
from lab2kit.checks import (
    check_accept,
    check_hill_climb,
    check_hill_climb_sideways,
    check_queens_cost,
    check_queens_neighbors,
    check_random_restart,
    check_simulated_annealing,
)


def ready(*results):
    """Checks that the necessary results exist and returns True if it does."""
    for result in results:
        if result is None:
            print("⏭  Skipped for now — run the experiment cell above first, "
                  "then come back.")
            return False
    return True


print("lab2kit loaded.")

### The 8-queens problem

Place 8 queens on an 8×8 chessboard so that none of them attacks another. Queens attack along rows, columns, and diagonals.

We will only ever consider boards with **exactly one queen per column**, so a board is a tuple of 8 numbers: `state[col]` is the row of the queen in column `col`. That representation makes column conflicts impossible by construction, leaving rows and diagonals to fix.

Here is a random board, with every attacking pair joined by a line.

In [ ]:
# PROVIDED
rng = np.random.default_rng(42) # This is your random number generator
random_board = tuple(int(row) for row in rng.integers(0, 8, size=8))
print("state:", random_board)
viz.show_board(random_board)

### Random guessing: a baseline

The simplest possible search is to keep guessing whole boards at random and keep the best one you have seen. The cell below does exactly that: 2,000 guesses, 50 times over.

There are $8^8 = 16{,}777{,}216$ boards and only 92 solutions, so guessing is a long shot. Let's see how it does.

In [ ]:
# PROVIDED
rng = np.random.default_rng(0)
solved_runs = 0
best_overall = 28  # the worst a board can be: every one of the 28 pairs attacking

for run_number in range(50):
    best_this_run = 28
    for guess in range(2000):
        board = tuple(int(row) for row in rng.integers(0, 8, size=8))
        pairs = viz.reference_cost(board)
        if pairs < best_this_run:
            best_this_run = pairs
    if best_this_run == 0:
        solved_runs += 1
    if best_this_run < best_overall:
        best_overall = best_this_run

print(f"random guessing: {solved_runs} of 50 runs solved 8-queens in 2,000 guesses")
print(f"best board found across all 50 runs had {best_overall} attacking pairs")

Guessing is hopeless because it throws away everything it learns. A board with one attacking pair tells you a lot, but guessing doesn't use this information.

**The key idea of this lab:** take a board you have and move to a *neighboring* board that is better. That is local search.

### The interface

`NQueens` and `TSP` are two classes with exactly the same method names, so an algorithm written against one works on the other without changing a line:

| method | meaning |
|---|---|
| `problem.random_state(rng)` | a random state to start from |
| `problem.cost(state)` | a number to **minimize** (0 attacking pairs is a solved board) |
| `problem.all_neighbors(state)` | every state one move away |
| `problem.random_neighbor(state, rng)` | one of those, chosen at random |
| `problem.is_goal(state)` | `True` for a perfect solution (n-queens only) |

Each problem also **counts its own work** as you go:

| counter | meaning |
|---|---|
| `problem.evaluations` | how many times `cost()` has been called |
| `problem.iterations` | how many times you asked for neighbors |

Those two counters will be used in Part 5.

Two more details to keep in mind:

* States are always tuples (Lists would cause problems!)
* Every function that needs randomness is handed an `rng` and uses that to make random choices.

## Part 2: Formulating n-queens

*About 45 minutes. Goal G1: formulate a problem for local search.*

### Task 1: `queens_cost`

Here you will build a function that counts how many pairs of queens attack each other on a board.

Local search needs a number to push downhill: this is the cost. A board with cost 0 is a solved board, and every algorithm in this lab is just a different way of walking towards cost 0.

Two queens, in columns $i < j$, attack each other when

$$\text{row}_i = \text{row}_j \qquad\text{(same row)}$$
or
$$|\text{row}_i - \text{row}_j| = j - i \qquad\text{(same diagonal)}.$$

The diagonal condition says: they attack if moving from column $i$ to column $j$ changes the row by exactly as much as it changes the column.

Worked example, `state = (1, 3, 0, 2)`:

| pair | rows | $|\Delta\text{row}|$ | $\Delta\text{col}$ | attacking? |
|---|---|---|---|---|
| (0, 1) | 1, 3 | 2 | 1 | no |
| (0, 2) | 1, 0 | 1 | 2 | no |
| (0, 3) | 1, 2 | 1 | 3 | no |
| (1, 2) | 3, 0 | 3 | 1 | no |
| (1, 3) | 3, 2 | 1 | 2 | no |
| (2, 3) | 0, 2 | 2 | 1 | no |

So `queens_cost((1, 3, 0, 2))` is 0: this is a solved 4-queens board.

**Your task.** Write `queens_cost(state)` returning the number of attacking **pairs**, counting each pair once. A queen never attacks itself.

In [ ]:
def queens_cost(state):
    """Return the number of pairs of queens that attack each other.

    Args:
        state: A tuple of length n. state[col] is the row of the queen in
            column col.

    Returns:
        The number of attacking pairs, as an int. 0 means the board is solved.
    """
    raise NotImplementedError  # your code here

In [ ]:
check_queens_cost(queens_cost)

<details><summary>Hint 1</summary>

You need to look at every *pair* of columns. That is two nested loops. The question to answer for each pair is a yes/no one, so you are counting how many times a condition holds.

</details>

<details><summary>Hint 2</summary>

If the outer loop runs `i` over all columns and the inner loop runs `j` over all columns, you will see each pair twice, once as `(2, 5)` and once as `(5, 2)`. Make the inner loop start just after `i` so that `j` is always greater than `i`.

</details>

<details><summary>Hint 3</summary>

For columns `i` and `j` with `i < j`, the column distance is `j - i`. The row distance is `abs(state[i] - state[j])`. 

</details>

### Task 2: `queens_neighbors`

Here you will build a function listing every board one move away from the current one.

To do hill climbing, we need to specify what we mean by neighbors. This definition is part of the search landscape.

Here is how we will define neighboring moves: A move takes one queen and slides it to a different row in its own column.

The neighbor function should build a new tuple from the current one. You can do this by slicing around the position you want to replace. Here is an example with 4 columns:

```python
state = (1, 3, 0, 2)
col = 2 #make move on col
row = 3 # change to row 3
state[:col] + (row,) + state[col + 1:]   # gives (1, 3, 3, 2)
```


**Your task:** Write `queens_neighbors(state)` returning a list of tuples: every board reached by moving exactly one queen to a different row of its own column. There are $n$ columns and $n - 1$ other rows in each, so the list has $n(n-1)$ entries -- 56 for an 8×8 board. The current board is not one of its own neighbors.

In [ ]:
def queens_neighbors(state):
    """Return every state reached by moving one queen within its own column.

    Args:
        state: A tuple of length n.

    Returns:
        A list of n * (n - 1) states, each one a tuple, not including state
        itself.
    """
    raise NotImplementedError  # your code here

In [ ]:
check_queens_neighbors(queens_neighbors)

<details><summary>Hint 1</summary>

You are producing one neighbor per (column, new row) combination. That is again two nested loops, but this time over columns and rows rather than over pairs of columns.

</details>

<details><summary>Hint 2 </summary>

Every combination counts except the ones where the "new" row is the row the queen is already in. Those are not moves, and including them would let a search sit still forever.

</details>

<details><summary>Hint 3 </summary>

Start with an empty list. Inside the loops you have a column and a row: use the slicing idiom above to build the new tuple, and append it to the list. Return the list at the end.

</details>

### What the landscape looks like

Now that you have both pieces, `NQueens` can be built out of them. The figure below is the whole neighborhood of one board: each square shows what the cost would become if you moved that column's queen to that row. The circles mark where the queens are now.

This is a picture of what a hill climber sees when it decides where to step.

In [ ]:
# PROVIDED
queens = None
try:
    queens = NQueens(8, queens_cost, queens_neighbors)
    viz.show_neighbor_costs(queens, random_board)
except NotImplementedError:
    queens = None
    print("⏭  Skipped for now — finish Tasks 1 and 2, then run this cell again.")

## Part 3: Hill climbing

*About 90 minutes.*

In hill climbing we **look at every neighbor, go to the best one, repeat until nothing is better.** (We are minimizing cost, so we are really descending, but we will stick with the standard name.)

### Spending a budget

Every algorithm from here on takes a third argument, `budget`: the number of cost evaluations it is allowed to spend. The problem counts them for you, so the loop condition is just

```python
while problem.evaluations < budget:
```

Put that at the top of your main loop. There is no more tracking necessary. We need a budget for two reasons: we want to prevent infinite loops, and we want a way to compare the efficiency of the algorithms (part 4).

This means that your loop can now end two ways. Hill climbing stops either because it reached a local minimum (a `return` inside the loop) or because the budget ran out (a `return` after it). Both are valid termination conditions.

### Task 3: `hill_climb`

We will build steepest-descent hill climbing.

This is the first working baseline every other algorithm will be measured against. Importantly, we will see when it works and when it doesn't.

Start at `problem.random_state(rng)`. At each step, compute the cost of every neighbor. If the cheapest neighbor is **strictly cheaper** than where you are, move there; otherwise stop and return the current state.

Important notes:

* **Strictly** better. If you also accept equal-cost moves you can walk back and forth across a flat region forever.
* When several neighbors tie for cheapest, **pick one uniformly at random using `rng`**. We don't want the success rate to depend on how we order neighboring states.

First, let's predict how hill-climbing will go before you implement it. We did discuss this in class, but entry your guess as a variable below.

In [ ]:
# From a random board, repeatedly move to the best neighboring board, 
#and stop when no neighbor is better. What fraction of runs do you think ends on a solved board? 
#(We discussed this in class, and a number is given in the text!)
hc_success_rate_8queens = None # replace with a number between 0 and 1


**Now for your task:** Write `hill_climb(problem, rng, budget)` that returns the a state once you stop.

In [ ]:
def hill_climb(problem, rng, budget):
    """Steepest-descent hill climbing with random tie-breaking.

    Args:
        problem: A problem offering cost, all_neighbors and random_state.
        rng: The generator to use for the start state and for tie-breaking.
        budget: How many cost evaluations this run may spend.

    Returns:
        The state reached when no neighbor is strictly better, or when the
        budget runs out.
    """
    raise NotImplementedError  # your code here

In [ ]:
check_hill_climb(hill_climb, queens)

<details><summary>Hint 1</summary>

you should have a loop like `while problem.evaluations < budget` that has a `return state` inside it. Each pass checks if there a strictly better neighbor. If yes, move and repeat. If no, return. Put a second `return` after the loop for the case where the budget ran out first.

</details>

<details><summary>Hint 2</summary>

Get the list of neighbors once, then build a parallel list of their costs. Use `min(costs)` is the best cost available, and you can compare it against the cost of the state you are standing on.

</details>

<details><summary>Hint 3</summary>

Collect the *positions* in the cost list that equal the minimum. loop over the costs with their index and keep the ones that match. Then choose one of those positions with `rng.integers(len(...))` and take the neighbor at that position.

</details>

<details><summary>Hint 4</summary>

You do not need to re-evaluate the state you are standing on every time round the loop. When you move, you already know the cost of where you are moving to: keep it in a variable.

</details>

### Task 4: `hill_climb_sideways`

Here you will build an algoirthm that implements hill climbing, but is allowed to step across flat ground.

Look again at the heat map in Part 1. A lot of squares tie with the current cost. A climber that refuses every tie stops the moment it reaches a flat shoulder, even when the way onward is just a few flat steps away.

When the best neighbor is *equal* in cost, move anyway, but only up to `max_sideways` times **in a row**. This means you should have a counter that resets to zero after any strictly improving move. If the best neighbor is worse, stop as before.

First, what do you think the success rate will be with sideways moves allowed? Record your answer as a variable below.

In [ ]:
sideways_success_rate_8queens = None # record your guess here

Let's start implementing. Start by copying your Task 3 solution. You should be able to just make a few small modifications.

**Task:** Write `hill_climb_sideways(problem, rng, budget, max_sideways=100)`.

In [ ]:
def hill_climb_sideways(problem, rng, budget, max_sideways=100):
    """Hill climbing that also accepts up to max_sideways equal-cost moves.

    Args:
        problem: A problem offering cost, all_neighbors and random_state.
        rng: The generator to use for the start state and for tie-breaking.
        budget: How many cost evaluations this run may spend.
        max_sideways: How many equal-cost moves may be made in a row before
            giving up. The count resets after any strict improvement.

    Returns:
        The state reached when the climb stops.
    """
    raise NotImplementedError  # your code here

In [ ]:
check_hill_climb_sideways(hill_climb_sideways, queens)

<details><summary>Hint 1</summary>

In Task 3 you returned whenever the best neighbor was not strictly better. Now that condition splits in two: strictly worse means stop, exactly equal means "maybe, if I still have sideways moves left".

</details>

<details><summary>Hint 2</summary>

Keep an integer next to your state. Increase it when you take an equal-cost move, and set it back to zero when you take an improving one.

</details>

<details><summary>Hint 3</summary>

Before taking a sideways move, check whether the counter has already reached `max_sideways`. If it has, return instead of moving.

</details>

### Task 5: `random_restart`

We will build a wrapper that keeps restarting a climber from new random boards until one of them lands on a solution.

Call `climber(problem, rng, budget)` to get a local minimum. If `problem.is_goal(state)` is true, return it. Otherwise go round again; the climber draws a fresh random start each time, as long as you keep passing it the same `rng`.

This is the first algorithm with no natural stopping point: if the climber never reaches a goal, the budget is the only thing that ends it. 

(The answer to this task is provided below.)

In [ ]:
# PROVIDED
def random_restart(problem, rng, budget, climber=hill_climb):
    """Restart climber from fresh random states until it reaches a goal."""
    state = None
    while problem.evaluations < budget:
        state = climber(problem, rng, budget)
        if problem.is_goal(state):
            return state
    return state

In [ ]:
check_random_restart(random_restart, queens)

### Experiment 1: how often does each one solve 8-queens?

*Provided code.* 200 random starts each for plain hill climbing and for hill climbing with up to 100 sideways moves, then 50 runs of random restarts. Every run gets its own seed, and every algorithm sees the same seeds.

In [ ]:
# PROVIDED
QUEENS_SEEDS = list(range(200))
RESTART_SEEDS = list(range(50))
QUEENS_BUDGET = 20000
RESTART_BUDGET = 100000


def make_queens(seed):
    """Build the 8-queens problem.

    The board is the same every time. What the seed changes is where each run
    starts from, and that comes from the generator, not from here.
    """
    return NQueens(8, queens_cost, queens_neighbors)


df_queens = None
try:
    climbers = {
        "hill climbing": hill_climb,
        "hill climbing + sideways": hill_climb_sideways,
    }
    df_climbers = run_all(climbers, make_queens, QUEENS_SEEDS, QUEENS_BUDGET)
    df_restarts = run_all({"random restarts": random_restart}, make_queens,
                          RESTART_SEEDS, RESTART_BUDGET)
    df_queens = pd.concat([df_climbers, df_restarts], ignore_index=True)
except NotImplementedError:
    print("⏭  Skipped for now — finish Tasks 3, 4 and 5, then run this cell again.")

In [ ]:
# PROVIDED
if ready(df_queens):
    viz.plot_success_rates(
        df_queens,
        reference={"hill climbing": 0.14, "hill climbing + sideways": 0.94},
    )
    summary = df_queens.groupby("algorithm").agg(
        runs=("seed", "count"),
        success_rate=("solved", "mean"),
        median_evaluations=("evaluations", "median"),
    )
    solved_only = df_queens[df_queens["solved"]]
    summary["median_evaluations_when_solved"] = solved_only.groupby("algorithm")[
        "evaluations"
    ].median()
    display(summary)

The dashed lines are the success rates *AIMA* Ch 4.1 (this week's reading) reports for these two algorithms on 8-queens: about 14% for steepest-ascent hill climbing and about 94% once up to 100 sideways moves are allowed.

With 200 runs, landing a few percentage points either side of those lines is exactly what sampling noise looks like. Landing far away is a signal worth chasing: re-read your Task 3 stopping condition (strictly better?) and your tie-breaking (actually random?). The check cells for Tasks 3 and 4 test both of those directly.

The table also reports the median number of evaluations spent. Keep that column in mind — Part 5 is about what happens when you take it seriously.

## Part 4: Simulated annealing

*About 60 minutes.*

Hill climbing fails for one reason: it will not go uphill, so it stops at the first local minimum. Restarts work around that by throwing the run away and starting over.

Simulated annealing takes a different line: it allows us to go uphill sometimes with some probability. The probability depends on a number called *Temperature* that we will vary during the run of the algorithm. As discussed in class, simulated anneasing comes from metallurgy. We are starting from a hot temp and cooling it slowly, so that the system (our state) can slowly settle down into a low-energy state. If we cool it too quickly, then the system would freeze at a random state: that is, it will become trapped in a local solution.

[Here is a popular science Youtube video that explains the concept](https://www.youtube.com/watch?v=I_0GBWCKft8).

### Task 6a: `accept`

**What you will build:** The Metropolis rule. This is the decision step at the core of SA.

**Here is how it works:** A candidate move changes the cost by `delta` (negative means better). A temperature `T` controls how tolerant we are:

$$P(\text{accept}) = \begin{cases} 1 & \delta \le 0 \\[2pt] e^{-\delta / T} & \delta > 0 \end{cases}$$

At high `T` the exponent is near zero and almost everything is accepted, so the search wanders and explores. At low `T` even a small uphill step is rejected, so the search is effectively greedy. Cooling from high to low means exploring first and refining later.

Plug in some numbers (or even graph it) get get a feel for how the equation behaves. For example, with $\delta = 1$ and $T = 1$, $e^{-1} \approx 0.37$. With the same $\delta$ and $T = 0.25$, $e^{-4} \approx 0.018$.

**Your task:** Write `accept(delta, T, rng)` returning a bool, `True` or `False` according to the Metropolis rule. Always accept when `delta <= 0`: if `delta >0 ` you will use an rng to make a probabilistic decision. When `T <= 1e-12`, treat the system as `frozen`, meaning we only accept if `delta <= 0`. Make sure not to divide by zero.

In [ ]:
def accept(delta, T, rng):
    """Decide whether to accept a move that changes the cost by delta.

    Args:
        delta: New cost minus current cost. Negative means the move improves.
        T: The current temperature.
        rng: The generator to draw the acceptance coin from.

    Returns:
        True if the move should be taken.
    """
    raise NotImplementedError  # your code here

In [ ]:

def accept(delta, T, rng):
    """Metropolis acceptance rule for a move that changes the cost by delta."""
    if delta <= 0:
        return True
    if T <= 1e-12:
        return False
    return bool(rng.random() < math.exp(-delta / T))

In [ ]:
check_accept(accept)

<details><summary>Hint 1 — the easy cases first</summary>

Two situations need no randomness at all: a move that does not make things worse, and a temperature so low that nothing uphill should ever be taken. Handle both with early returns, then only the interesting case is left.

</details>

<details><summary>Hint 2 — turning a probability into a decision</summary>

`rng.random()` gives a number uniformly in [0, 1). It is below `p` with probability exactly `p`. That is how you flip a biased coin.

</details>

<details><summary>Hint 3 — check the sign</summary>

A bigger `delta` is a worse move, so it must be accepted *less* often.

</details>

### Task 6b: `simulated_annealing`

**What you'll build.** The annealing loop.

Unlike hill climbing, this looks at **one** random neighbor per step rather than all of them. One step costs one evaluation instead of 56. Part 5 is about what that difference is worth.

**What you need to know.** Start from a random state and set `T = T0`. Each iteration:

1. draw `candidate = problem.random_neighbor(state, rng)`;
2. compute `delta` = candidate's cost − current cost;
3. if `accept(delta, T, rng)`, the candidate becomes the current state;
4. cool: `T = alpha * T`;
5. if `T` has fallen below `T_min`, set it back to `T0`.

Step 5 means the schedule never finishes on its own: once it freezes it starts again, so the search keeps working right up to the budget. The defaults `T0=1.0, alpha=0.9995` are calibrated for this lab's problems and budgets.

**Your task.** Write `simulated_annealing(problem, rng, budget, T0=1.0, alpha=0.9995, T_min=1e-3)` and return the current state.

In [ ]:
def simulated_annealing(problem, rng, budget, T0=1.0, alpha=0.9995, T_min=1e-3):
    """Simulated annealing with geometric cooling and a restarting schedule.

    Args:
        problem: A problem offering cost, random_neighbor and random_state.
        rng: The generator for the start state, the neighbors and accept.
        budget: How many cost evaluations this run may spend.
        T0: The starting temperature, and the one the schedule resets to.
        alpha: The cooling factor applied once per iteration, just under 1.
        T_min: When T falls below this, the schedule restarts at T0.

    Returns:
        The current state when the budget runs out.
    """
    raise NotImplementedError  # your code here

In [ ]:
check_simulated_annealing(simulated_annealing, queens)

<details><summary>Hint 1</summary>

`problem.random_neighbor(state, rng)` returns a single state. Remember that SA selects a random state at each loop: no need to look at all neighbors.

</details>

<details><summary>Hint 2</summary>

You need the current state's cost to compute `delta`. Evaluate it once before the loop and update it whenever you accept a move; otherwise you spend two evaluations per step to do the work of one, and your budget goes half as far.

</details>

<details><summary>Hint 3</summary>

If `accept` says yes, the candidate has to become the current state for the next iteration.

</details>

<details><summary>Hint 4</summary>

Cooling happens every iteration, accepted or not. After cooling, compare `T` with `T_min` and reset it to `T0` if it has dropped below.

</details>

### Traveling salesperson problem

The [Travelling salesperson problem](https://en.wikipedia.org/wiki/Travelling_salesman_problem) is a classic problem in computer science. You wrote the simulated annealing code for the n-Queens problem, but will it work on the TSP? Below the provided code will run your `simulated_annealing` on the TSP. The problem will generate the coordinates of 

*Provided code.* Below, your `simulated_annealing` will be used to solve 50-city travelling salesperson problem. A state is now a tour (an order to visit the cities in) and a neighbor reverses a segment of it (called a [two-opt](https://en.wikipedia.org/wiki/2-opt). The cost is the length of the closed tour.

Let's see how your function does!

In [ ]:
# PROVIDED
tsp_demo = TSP(50, seed=0)
demo = None
try:
    demo = run_one(simulated_annealing, tsp_demo, 100_000, 0, "annealing")
    print(f"tour length: {demo['trace'][0][2]:.2f} at the start"
          f"  →  {demo['final_cost']:.2f} at the end")
    viz.show_tour_progress(tsp_demo, demo, n_panels=4)
except NotImplementedError:
    print("⏭  Skipped for now — finish Task 6b, then run this cell again.")

That is the same function you tested on 8-queens two cells ago.

This is what the interface bought us. You wrote an algorithm against `cost` and `random_neighbor`; anything that can supply those is now something you can search.

## Part 5: Benchmarking

*About 45 minutes. Goal G3: compare algorithms fairly at equal budget.*

"Which algorithm is better" is not a question until you say *better per what*. Three candidates:

* **Iterations** — how many times round the main loop. Easy to count, but one hill-climbing iteration evaluates all 1,224 two-opt neighbors of a 50-city tour, while one annealing iteration evaluates one. Comparing these as equals compares very different amounts of work.
* **Wall-clock time** — what you actually wait for, but it depends on your laptop, your Python version, and what else is running. Not reproducible.
* **Cost evaluations** — the number of `problem.cost(...)` calls. In this lab, and in most of local search, evaluating the objective is where the time goes, and the count does not depend on the machine. This is the unit we have been budgeting in all along.

Your problems have been counting both numbers this whole time, in `problem.evaluations` and `problem.iterations`.

Let's first make a prediction. Do you think that given the same budget, HC with restarts or SA will perform better? Record your answer below, and a brief answer (1-2 sentences) why:

In [ ]:
tsp_winner_equal_evaluations = 'annealing' # write annealing or restarts
why_your_answer = ""

# RECORDING YOUR PREDICTIONS
predictions = record_predictions(
    hc_success_rate_8queens=hc_success_rate_8queens,     
    sideways_success_rate_8queens=sideways_success_rate_8queens, 
    tsp_winner_equal_evaluations=tsp_winner_equal_evaluations,  
    why=why_your_answer,    
)

### Experiment 2: annealing versus restarts on TSP

*Provided code.* Twenty 50-city instances, a different one per seed. Both algorithms get the **same 100,000 cost evaluations** on each instance. This takes about half a minute.

In [ ]:
# PROVIDED
TSP_SEEDS = list(range(20))
TSP_BUDGET = 100_000


def make_tsp(seed):
    """Build a 50-city instance. Each seed gives a different set of cities."""
    return TSP(50, seed=seed)


df_tsp = None
try:
    tsp_algorithms = {
        "annealing": simulated_annealing,
        "restarts": random_restart,
    }
    df_tsp = run_all(tsp_algorithms, make_tsp, TSP_SEEDS, TSP_BUDGET)
except NotImplementedError:
    print("⏭  Skipped for now — finish Tasks 5 and 6b, then run this cell again.")

In [ ]:
# PROVIDED
if ready(df_tsp):
    viz.plot_budget_reversal(df_tsp)
    display(
        df_tsp.groupby("algorithm").agg(
            median_tour_length=("final_cost", "median"),
            median_evaluations=("evaluations", "median"),
        )
    )

Both panels show the same twenty runs of each algorithm. The only difference is what the horizontal axis counts: on the left, turns of the main loop; on the right, cost evaluations. The shaded band is the interquartile range across seeds.

**Look at both panels. Does your conclusion depend on the x-axis? Why?**

You will answer that in Part 6. Before you do, work out roughly how many cost evaluations one iteration of each algorithm spends.

In [ ]:
# PROVIDED
if ready(df_queens, df_tsp):
    save_results(pd.concat([df_queens, df_tsp], ignore_index=True))

In [ ]:
# PROVIDED
if ready(df_queens, df_tsp):
    display(reveal(predictions, df_queens, df_tsp))

## Part 6: Reflect

*Spend about 30 min*

Answer each of the three questions below in 3 to 6 sentences, in the markdown cell that follows it.

**1.** Why do sideways moves change the success rate on n-queens so much? Refer to what you saw in the neighborhood heat map in Part 2.

*Your answer here.*

**2.** Compare your predictions with the reveal table above. Where were you wrong, and what were you misjudging?

*Your answer here.*

**3.** Explain the budget reversal. Which unit of budget is the right one for comparing these two algorithms, and when might a different unit be the right one instead?

*Your answer here.*

## Looking forward

Everything in this lab searched a space of tuples, where "neighbor" meant a *discrete* edit: move a queen, reverse a segment. Suppose instead your state is a vector of real numbers and the cost is a smooth function of them. The neighbors are now every point in a small ball around you, and you cannot enumerate them, but you can estimate which direction goes downhill fastest, and take a step that way. Hill climbing in a continuous space, with the step direction estimated from the local slope, is **gradient descent**. We will use gradient descent extensively in the latter half of the course.

---

**Additional Work:** `lab2_extensions.ipynb` will contain some more advanced exercises you can try for additional credit.

**Before you submit:** run this notebook top to bottom one last time (Kernel $\to$ Restart Kernel and Run All Cells), check that `results.csv` and `predictions.json` are in the folder, and commit all three files.